## Imports

In [1]:
import pandas as pd

##  Load the raw data

In [3]:
df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_raw.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 10000 rows


,reviewId,userName,review_text,rating,thumbs_up,review_date,reviewCreatedVersion,appVersion,replyContent
0,614974df-eef0-4b87-887c-0d52ea3b4b12,Martyn Boyd,does exactly what I need it to do with more st...,5,0,2026-07-19 02:24:39,10.138,10.138,NaN
1,ce70ff3b-a835-4f82-898f-5fdf3749adb6,Edward Krosendijk,nice,5,0,2026-07-18 23:49:34,10.138,10.138,NaN
2,75a04d4f-9d10-4961-be06-6cca1e87302d,Steven Carey,Does what it says on the tin.,5,0,2026-07-18 23:43:32,10.138,10.138,NaN
3,33a38df1-6cc8-4b87-bd2e-c2a83c4af526,Matt Kelly,Good app but frustrating that I can't see my r...,5,0,2026-07-18 22:41:09,10.138,10.138,Hi there. Thank you for your feedback. We're s...
4,3faba9bd-8588-4885-92b2-c96552e37fc3,Ashu Kalonia,great,5,0,2026-07-18 22:27:19,10.138,10.138,NaN


## Check for missing values

In [4]:
df.isnull().sum()

reviewId                   0
userName                   2
review_text                0
rating                     0
thumbs_up                  0
review_date                0
reviewCreatedVersion    1749
appVersion              1749
replyContent            8547
dtype: int64

##  Drop columns we don't need

In [5]:
df = df.drop(columns=["userImage"], errors="ignore")
df.head()

,reviewId,userName,review_text,rating,thumbs_up,review_date,reviewCreatedVersion,appVersion,replyContent
0,614974df-eef0-4b87-887c-0d52ea3b4b12,Martyn Boyd,does exactly what I need it to do with more st...,5,0,2026-07-19 02:24:39,10.138,10.138,NaN
1,ce70ff3b-a835-4f82-898f-5fdf3749adb6,Edward Krosendijk,nice,5,0,2026-07-18 23:49:34,10.138,10.138,NaN
2,75a04d4f-9d10-4961-be06-6cca1e87302d,Steven Carey,Does what it says on the tin.,5,0,2026-07-18 23:43:32,10.138,10.138,NaN
3,33a38df1-6cc8-4b87-bd2e-c2a83c4af526,Matt Kelly,Good app but frustrating that I can't see my r...,5,0,2026-07-18 22:41:09,10.138,10.138,Hi there. Thank you for your feedback. We're s...
4,3faba9bd-8588-4885-92b2-c96552e37fc3,Ashu Kalonia,great,5,0,2026-07-18 22:27:19,10.138,10.138,NaN


## Handle missing review text

In [6]:
before = len(df)
df = df.dropna(subset=["review_text"])
df = df[df["review_text"].str.strip() != ""]
after = len(df)
print(f"Dropped {before - after} rows with empty review text")
print(f"Remaining rows: {after}")

Dropped 0 rows with empty review text
Remaining rows: 10000


## Remove duplicate reviews

In [7]:
before = len(df)
df = df.drop_duplicates(subset=["reviewId"])
after = len(df)
print(f"Dropped {before - after} duplicate rows")

Dropped 0 duplicate rows


## Handle Missing Data

In [11]:
# userName: fill missing with "Anonymous" since it's not used in analysis anyway
df["userName"] = df["userName"].fillna("Anonymous")

# reviewCreatedVersion / appVersion: fill with "Unknown" — 
# we're not dropping rows just because we don't know the app version
df["reviewCreatedVersion"] = df["reviewCreatedVersion"].fillna("Unknown")
df["appVersion"] = df["appVersion"].fillna("Unknown")

# replyContent: create a new column that tracks whether Revolut replied at all —
# this turns a "missing value problem" into a useful feature
df["got_reply"] = df["replyContent"].notnull()

# keep replyContent as-is (with NaN) — we're not filling it with fake text,
# NaN correctly represents "no reply was given"

print(df[["userName", "reviewCreatedVersion", "appVersion", "got_reply"]].isnull().sum())
print(f"\nReply rate: {df['got_reply'].mean():.1%}")

userName                0
reviewCreatedVersion    0
appVersion              0
got_reply               0
dtype: int64

Reply rate: 14.5%


## Convert date column to proper datetime

In [8]:
df["review_date"] = pd.to_datetime(df["review_date"])
df["review_date"].head()

0   2026-07-19 02:24:39
1   2026-07-18 23:49:34
2   2026-07-18 23:43:32
3   2026-07-18 22:41:09
4   2026-07-18 22:27:19
Name: review_date, dtype: datetime64[us]

## Add a few useful derived columns

In [9]:
df["year"] = df["review_date"].dt.year
df["month"] = df["review_date"].dt.month
df["review_length"] = df["review_text"].str.len()

df[["review_date", "year", "month", "review_length"]].head()

,review_date,year,month,review_length
0,2026-07-19 02:24:39,2026,7,60
1,2026-07-18 23:49:34,2026,7,4
2,2026-07-18 23:43:32,2026,7,29
3,2026-07-18 22:41:09,2026,7,340
4,2026-07-18 22:27:19,2026,7,5


## Final check before saving

In [13]:
df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_clean.csv", index=False)
print("Saved to E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_clean.csv")

Saved to E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_clean.csv
